In [43]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

In [44]:
import joblib
model = joblib.load("lr_model.pkl")
tfidf = joblib.load("tfidf_vector.pkl")
le = joblib.load("label_encoder.pkl")
train=pd.read_csv("train_preprocessed.csv")
test=pd.read_csv("test_preprocessed.csv")

In [45]:
train['text'].head()

0    anniversary special buy one get one free loyal...
1    amazon used new device 5000 refund processed. ...
2    google inquiry hi, following up google applica...
3    digital ritual experience creation cross-cultu...
4    post moved programming help trending cooking 2...
Name: text, dtype: object

In [ ]:
HIGH = [
   
    r"\bexpires?\s+at\s+\d{1,2}:\d{2}\b",   
    r"\btoken\b.*\bexpir\b",                 
    r"\bvalid\s+for\s+\d+\s+minutes?\b",      
    r"\bwithin\s+\d+\s+(hours?|mins?)\b",     
    r"\bfinal\s+hours?\b",                    
    r"\beffective\s+immediately\b",           
    r"\bimmediately\b",                       
    r"\baction\s+required\b",                 
    r"\bpayment\s+fail(ed|ure)\b",           
    r"\bcouldn.?t\s+be\s+delivered\b",       
    r"\bupdate\s+now\b",                      
    r"\boverdue\b",                           
    r"\bcompromised\b",                       
    r"\bunauthorized\b",                      
    r"\baccount\s+locked\b",                  
    r"\bsuspicious\s+activity\b",             
    r"\burgent\b",                            
    r"\basap\b",r"\bexpires?\s+at\s+\d{1,2}:\d{2}\b",    
    r"\btoken\b",                              
    r"\bauto.cancel\b",                        
    r"\bsingle.use\b",                         
    r"\bdo\s+not\s+share\b",                  
    r"\bsubmit\s+within\b",                   
    r"\bsession\s+will\b",                 
    r"\bimmediate\b",                          
    r"\bclaim\b.*\bnow\b",                    
    r"\bcomplete\s+within\b",                 
    r"\bverify\s+now\b",                      
    r"\bact\s+now\b",                         
    r"\blimited\s+time\b",                    
    r"\btrack\b.*\bupdate\b",                 
    r"\bfailed\b",                            
    r"\bblocked\b",                           
    r"\bsuspended\b",                              
]

MEDIUM = [
    
    r"\benter\s+\d{4,8}\b",                  
    r"\bverification\s+code\b",              
    r"\baccess\s+code\b",                    
    r"\bpassword\s+reset\b",                 
    r"\baccount\s+recovery\b",               
    r"\btwo.step\b",                        
    r"\bsecurity\s+upgrade\b",              
    r"\bconfirm\s+your\b",                  
    r"\bpending\s+(friend|request)\b",       
    r"\baccept\s+or\s+reject\b",            
    r"\bjoin\s+request\b",                   
    r"\bnew\s+features?\s+available\b",      
    r"\bstatement\s+available\b",            
    r"\border\s+confirm\b",                  
    r"\bpayment\s+confirm\b",               
    r"\bsubscription\s+change\b",
    r"\benter\s+\d{4,8}\b",
    r"\bverification\s+code\b",
    r"\baccess\s+code\b",
    r"\bpassword\s+reset\b",
    r"\baccount\s+recovery\b",
    r"\btwo.step\b",
    r"\bsecurity\s+upgrade\b",
    r"\bconfirm\s+your\b",
    r"\bpending\b",
    r"\baccept\s+or\s+reject\b",
    r"\bjoin\s+request\b",
    r"\bnew\s+features?\s+available\b",
    r"\bstatement\s+available\b",
    r"\bpayment\s+applied\b",
    r"\bnew\s+hours\b",                       
    r"\binstall\s+now\b",                     
    r"\bsecurity\s+update\b",                 
    r"\bnew\s+features\b",           
]

LOW = [
    
    r"\bdiscount\b", r"\boff\b", r"\bsale\b",
    r"\bpromo\b", r"\bdeal\b", r"\boffer\s+code\b",
    r"\bshop\s+now\b", r"\bbuy\s+now\b",
    r"\bnewsletter\b", r"\bunsubscribe\b",
    r"\bno\s+action\s+needed\b", r"\bannouncement\b",
    r"\bnew\s+(post|thread|reply|comment)\b",
    r"\bcheck\s+it\s+out\b", r"\bview\s+thread\b",
    r"\bcommunity\b", r"\bdiscussion\b",
    r"\bnew\s+follower\b", r"\bmentioned\s+you\b",
    r"\bbirthday\b", r"\banniversary\b",
    r"\bmemories\b", r"\bchecked\s+in\b",
    r"\bnearby\b", r"\bfollowing\b",
    
    r"\bmaintenance\b", r"\bscheduled\b",
    r"\bchangelog\b", r"\bno\s+action\b",
]

CATEGORY_DEFAULT = {
    "forum"       : "Low",     
    "promotions"  : "Low",     
    "social_media": "Low",     
    "spam"        : "Low",     
    "updates"     : "Low",     
    "verify_code" : "Medium",  
}

In [ ]:
def rule_based_urgency(text: str, category: str = "") -> str:
    text = str(text).lower()

    high_score   = sum(1 for p in HIGH   if re.search(p, text))
    medium_score = sum(1 for p in MEDIUM if re.search(p, text))
    low_score    = sum(1 for p in LOW    if re.search(p, text))

    
    if high_score > 0:
        return "High"
    
    elif medium_score > 0 and medium_score >= low_score:
        return "Medium"
    elif low_score > 0:
        return "Low"
    else:
    
        return CATEGORY_DEFAULT.get(category.lower(), "Low")


In [61]:
import re

train["urgency"] = train.apply(
    lambda row: rule_based_urgency(row["text"], row["category"]), axis=1
)

In [62]:
print("\nUrgency distribution:")
print(train["urgency"].value_counts())


Urgency distribution:
urgency
Low       6373
Medium    1855
High      1845
Name: count, dtype: int64


In [63]:
res=pd.read_csv("results_10k.csv")

In [64]:
y_true = train["urgency"].str.lower().str.strip()
y_pred = res["pred_urgency"].str.lower().str.strip()

In [65]:
print(f"Accuracy : {accuracy_score(y_true, y_pred):.2%}")
print(classification_report(y_true, y_pred, labels=["high", "medium", "low"]))

Accuracy : 54.16%
              precision    recall  f1-score   support

        high       0.93      0.35      0.51      1845
      medium       0.17      0.29      0.22      1855
         low       0.68      0.67      0.68      6373

    accuracy                           0.54     10073
   macro avg       0.59      0.44      0.47     10073
weighted avg       0.63      0.54      0.56     10073



In [66]:
category_urgency = train.groupby(["category", "urgency"]).size().unstack(fill_value=0)
print(category_urgency)

urgency       High   Low  Medium
category                        
forum          119  1341     112
promotions     212  1551       2
social_media     0  1606      83
spam           884   829       6
updates        205  1043     311
verify_code    425     3    1341


In [67]:
category_urgency = res.groupby(["category", "pred_urgency"]).size().unstack(fill_value=0)
print(category_urgency)

pred_urgency  high   low  medium
category                        
forum          111   914     547
promotions      23  1667      75
social_media     2   583    1104
spam           376   606     737
updates         34  1305     220
verify_code    150  1159     460


In [ ]:
print(rule_based_urgency("token expires at 09:45", "verify_code"))
print(rule_based_urgency("new follower added you", "social_media"))
print(rule_based_urgency("enter 924578 to verify", "verify_code"))

High
Low
Medium


In [69]:
train.to_csv("urgency_rule.csv", index=False)